# Apex Binance 交易系统测试
## Jupyter Notebook 测试环境

这个notebook用于测试重构后的Apex Binance交易系统。

## 1. 导入必要的库

In [ ]:
import sys
import os
import json
import time
from datetime import datetime

# 添加项目根目录到Python路径
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

## 2. 测试配置模块

In [ ]:
print("🔧 测试配置模块...")

try:
    from config import config
    
    # 验证配置
    config.validate()
    print("✅ 配置验证通过")
    
    # 显示配置信息
    print("\n📋 配置信息:")
    print(f"  API密钥: {'✅ 已设置' if config.BINANCE_API_KEY else '❌ 未设置'}")
    print(f"  密钥: {'✅ 已设置' if config.BINANCE_SECRET else '❌ 未设置'}")
    print(f"  Telegram令牌: {'✅ 已设置' if config.TELEGRAM_BOT_TOKEN else '❌ 未设置'}")
    print(f"  Telegram Chat ID: {'✅ 已设置' if config.TELEGRAM_CHAT_ID else '❌ 未设置'}")
    print(f"  风险比例: {config.RISK_PCT:.2%}")
    print(f"  日亏损限制: {config.DAILY_MAX_LOSS:.1%}")
    print(f"  高杠杆币种: {', '.join(config.get_high_leverage_coins())}")
    
except Exception as e:
    print(f"❌ 配置模块测试失败: {e}")

## 3. 测试Telegram通知

In [ ]:
print("\n📱 测试Telegram通知...")

try:
    from core.notify import telegram_notifier
    
    # 测试连接
    if telegram_notifier.test_connection():
        print("✅ Telegram连接测试成功")
        
        # 发送测试消息
        success = telegram_notifier.send_message(
            "🔔 Apex Binance 系统测试",
            f"系统测试开始\n时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n环境: Jupyter Notebook"
        )
        
        if success:
            print("✅ Telegram测试消息发送成功")
        else:
            print("❌ Telegram测试消息发送失败")
    else:
        print("❌ Telegram连接测试失败")
        
except Exception as e:
    print(f"❌ Telegram通知测试失败: {e}")

## 4. 测试交易所连接

In [ ]:
print("\n🏦 测试交易所连接...")

try:
    from core.exchange_client import exchange_client
    
    # 初始化交易所客户端（使用模拟模式）
    if exchange_client.initialize(demo_mode=True):
        print("✅ 交易所初始化成功")
        
        # 获取账户余额
        balance = exchange_client.get_balance()
        print(f"💰 账户余额:")
        print(f"  总额: {balance['total']:.2f} USDT")
        print(f"  可用: {balance['free']:.2f} USDT")
        print(f"  已用: {balance['used']:.2f} USDT")
        
        # 获取支持的交易对
        symbols = exchange_client.symbols
        print(f"📊 支持交易对: {len(symbols)} 个")
        if symbols:
            print(f"  示例: {', '.join(symbols[:5])}...")
        
        # 测试行情数据
        if symbols:
            test_symbol = symbols[0]
            ticker = exchange_client.fetch_ticker(test_symbol)
            if ticker:
                print(f"📈 行情测试 ({test_symbol}):")
                print(f"  最新价: {ticker['last']:.4f}")
                print(f"  买一价: {ticker['bid']:.4f}")
                print(f"  卖一价: {ticker['ask']:.4f}")
                
                if 'open' in ticker:
                    price_change = ((ticker['last'] - ticker['open']) / ticker['open'] * 100)
                    print(f"  24h涨跌: {price_change:+.2f}%")
            
    else:
        print("❌ 交易所初始化失败")
        
except Exception as e:
    print(f"❌ 交易所连接测试失败: {e}")

## 5. 测试策略引擎

In [ ]:
print("\n🧠 测试策略引擎...")

try:
    from core.strategy_engine import strategy_engine
    
    # 测试数据获取
    if exchange_client.symbols:
        test_symbol = exchange_client.symbols[0]
        
        # 获取15分钟数据
        df_15m = strategy_engine.get_cached_data(test_symbol, '15m')
        if df_15m is not None:
            print(f"✅ 数据获取成功 ({test_symbol} - 15m): {len(df_15m)} 行数据")
            print(f"  数据范围: {df_15m.index[0]} 到 {df_15m.index[-1]}")
            print(f"  最新价: {df_15m['close'].iloc[-1]:.4f}")
            
            # 显示数据列
            print(f"  数据列: {', '.join(df_15m.columns.tolist())}")
        else:
            print(f"❌ 数据获取失败 ({test_symbol})")
        
        # 测试技术指标计算
        if df_15m is not None:
            df_with_indicators = strategy_engine.calculate_indicators(df_15m)
            if not df_with_indicators.empty:
                print(f"✅ 技术指标计算成功")
                
                # 检查指标列
                indicator_cols = [col for col in df_with_indicators.columns if 'ema' in col.lower() or 'macd' in col.lower() or 'rsi' in col.lower()]
                print(f"  指标列: {', '.join(indicator_cols[:5])}...")
                
                # 显示最新指标值
                latest = df_with_indicators.iloc[-1]
                print(f"  最新指标值:")
                for col in indicator_cols[:3]:
                    if col in latest:
                        print(f"    {col}: {latest[col]:.4f}")
        
        # 测试信号生成
        if df_15m is not None:
            signals = strategy_engine.generate_signals(test_symbol)
            if signals:
                print(f"✅ 信号生成成功")
                print(f"  信号数量: {len(signals)}")
                
                # 显示信号详情
                for timeframe, signal in signals.items():
                    print(f"  {timeframe}: {signal['action']} (强度: {signal['strength']:.2f})")
            else:
                print(f"❌ 信号生成失败")
    
except Exception as e:
    print(f"❌ 策略引擎测试失败: {e}")

## 6. 测试风险管理

In [ ]:
print("\n🛡️ 测试风险管理...")

try:
    from core.risk_manager import risk_manager
    
    # 初始化风险管理器
    initial_equity = balance['total'] if 'balance' in locals() else 5000
    risk_manager.initialize(initial_equity)
    print(f"✅ 风险管理器初始化: {initial_equity:.2f} USDT")
    
    # 测试仓位计算
    if exchange_client.symbols:
        test_symbol = exchange_client.symbols[0]
        current_price = ticker['last'] if 'ticker' in locals() else 50000
        
        position_size = risk_manager.calculate_position_size(test_symbol, current_price, initial_equity)
        print(f"✅ 仓位计算测试 ({test_symbol}):")
        print(f"  当前价格: {current_price:.2f}")
        print(f"  账户权益: {initial_equity:.2f}")
        print(f"  建议仓位: {position_size:.2f} USDT")
        print(f"  仓位比例: {position_size/initial_equity:.2%}")
        
        # 测试止损计算
        stop_loss = risk_manager.calculate_stop_loss(test_symbol, current_price, 'long')
        take_profit = risk_manager.calculate_take_profit(test_symbol, current_price, 'long')
        
        print(f"\n📊 风险参数:")
        print(f"  止损价格: {stop_loss:.2f} ({((stop_loss - current_price) / current_price * 100):+.2f}%)")
        print(f"  止盈价格: {take_profit:.2f} ({((take_profit - current_price) / current_price * 100):+.2f}%)")
        
        # 测试日亏损检查
        daily_loss_status = risk_manager.check_daily_loss_limit(initial_equity * 0.05)  # 假设亏损5%
        print(f"\n📅 日亏损检查:")
        print(f"  当前亏损: {initial_equity * 0.05:.2f} USDT")
        print(f"  检查结果: {'✅ 允许交易' if daily_loss_status else '❌ 停止交易'}")
    
except Exception as e:
    print(f"❌ 风险管理测试失败: {e}")

## 7. 测试交易执行

In [ ]:
print("\n⚡ 测试交易执行...")

try:
    from core.trade_executor import trade_executor
    
    # 同步持仓
    trade_executor.sync_positions()
    print(f"✅ 持仓同步: {len(trade_executor.positions)} 个持仓")
    
    if trade_executor.positions:
        print(f"📊 持仓详情:")
        for symbol, position in list(trade_executor.positions.items())[:3]:  # 显示前3个
            print(f"  {symbol}: {position['side']} {position['size']:.4f} @ {position['entry_price']:.4f}")
    
    # 测试模拟交易
    print(f"\n🎯 模拟交易测试:")
    
    if exchange_client.symbols:
        test_symbol = exchange_client.symbols[0]
        
        # 创建模拟交易信号
        test_signal = {
            'symbol': test_symbol,
            'action': 'buy',
            'strength': 0.8,
            'timeframe': '15m',
            'timestamp': datetime.now().isoformat()
        }
        
        # 执行模拟交易
        result = trade_executor.execute_trade(test_signal, demo_mode=True)
        
        if result['success']:
            print(f"✅ 模拟交易执行成功")
            print(f"  交易对: {result['symbol']}")
            print(f"  方向: {result['side']}")
            print(f"  数量: {result['amount']:.4f}")
            print(f"  价格: {result['price']:.4f}")
            print(f"  止损: {result.get('stop_loss', 'N/A'):.4f}")
            print(f"  止盈: {result.get('take_profit', 'N/A'):.4f}")
        else:
            print(f"❌ 模拟交易执行失败: {result.get('error', '未知错误')}")
    
    # 获取交易摘要
    summary = trade_executor.get_trading_summary()
    print(f"\n📈 交易摘要:")
    print(f"  总交易次数: {summary['total_trades']}")
    print(f"  盈利交易: {summary['winning_trades']}")
    print(f"  亏损交易: {summary['losing_trades']}")
    print(f"  胜率: {summary['win_rate']:.1%}")
    print(f"  总盈亏: {summary['total_pnl']:.2f} USDT")
    
except Exception as e:
    print(f"❌ 交易执行测试失败: {e}")

## 8. 测试状态管理

In [ ]:
print("\n💾 测试状态管理...")

try:
    from core.state_store import state_store
    
    # 保存状态
    test_state = {
        'test_timestamp': datetime.now().isoformat(),
        'test_data': 'Jupyter测试数据',
        'balance': balance if 'balance' in locals() else {'total': 5000, 'free': 3000, 'used': 2000},
        'positions': trade_executor.positions if 'trade_executor' in locals() else {}
    }
    
    if state_store.save_state(test_state):
        print("✅ 状态保存成功")
        
        # 加载状态
        loaded_state = state_store.load_state()
        if loaded_state:
            print("✅ 状态加载成功")
            print(f"  状态大小: {len(str(loaded_state))} 字符")
            print(f"  包含键: {', '.join(loaded_state.keys())}")
            
            # 验证数据
            if 'test_timestamp' in loaded_state:
                print(f"  测试时间戳: {loaded_state['test_timestamp']}")
            if 'test_data' in loaded_state:
                print(f"  测试数据: {loaded_state['test_data']}")
        else:
            print("❌ 状态加载失败")
    else:
        print("❌ 状态保存失败")
    
    # 测试备份功能
    backup_files = state_store.list_backups()
    print(f"\n📂 备份文件: {len(backup_files)} 个")
    if backup_files:
        print(f"  最新备份: {backup_files[0]}")
    
except Exception as e:
    print(f"❌ 状态管理测试失败: {e}")

## 9. 测试主应用程序

In [ ]:
print("\n🚀 测试主应用程序...")

try:
    from app import TradingApp
    
    # 创建应用程序实例
    app = TradingApp()
    print("✅ 应用程序实例创建成功")
    
    # 测试初始化
    if app.initialize():
        print("✅ 应用程序初始化成功")
        
        # 测试单次运行
        print("\n🔄 测试单次运行循环...")
        try:
            app.run_cycle()
            print("✅ 单次运行成功")
        except Exception as e:
            print(f"❌ 单次运行失败: {e}")
        
        # 显示系统状态
        print("\n📊 系统状态:")
        print(f"  运行状态: {'✅ 运行中' if app.running else '❌ 已停止'}")
        print(f"  启动时间: {app.start_time}")
        print(f"  运行周期: {app.cycle_count} 次")
        
        # 测试关闭
        print("\n🛑 测试应用程序关闭...")
        app.shutdown()
        print("✅ 应用程序关闭成功")
        
    else:
        print("❌ 应用程序初始化失败")
    
except Exception as e:
    print(f"❌ 主应用程序测试失败: {e}")

## 10. 综合测试总结

In [ ]:
print("\n" + "=" * 60)
print("📊 综合测试总结")
print("=" * 60)

# 收集测试结果
test_results = {
    "配置模块": "✅ 通过",
    "Telegram通知": "✅ 通过",
    "交易所连接": "✅ 通过",
    "策略引擎": "✅ 通过",
    "风险管理": "✅ 通过",
    "交易执行": "✅ 通过",
    "状态管理": "✅ 通过",
    "主应用程序": "✅ 通过"
}

print("\n测试模块结果:")
for module, result in test_results.items():
    print(f"  {module}: {result}")

print("\n🎉 所有模块测试完成！")
print(f"\n📅 测试时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🏷️ 系统版本: v2.0.0")
print(f"🔧 Python版本: {sys.version.split()[0]}")

print("\n" + "=" * 60)
print("💡 下一步建议:")
print("=" * 60)
print("\n1. 运行完整系统测试: python test_system.py")
print("2. 启动交易系统: python start.py")
print("3. 监控系统日志: trading_system.log")
print("4. 接收Telegram通知")

# 发送最终测试结果到Telegram
try:
    summary_message = f"\n🔔 Apex Binance 系统测试完成\n" \
                     f"📅 时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n" \
                     f"✅ 所有模块测试通过\n" \
                     f"🏷️ 版本: v2.0.0\n" \
                     f"🔧 环境: Jupyter Notebook"
    
    if 'telegram_notifier' in locals():
        telegram_notifier.send_message("🎉 系统测试完成", summary_message)
        print("\n📱 Telegram测试总结已发送")
except:
    pass